<a href="https://colab.research.google.com/github/stanley-zhou/CIS5500_Final_Project_Group15/blob/main/notebooks/01_imdb_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Path to all files downloaded from imdb website
BASE_DIR = f"/content/drive/Shareddrives/CIS5500/IMDb_Datasets"
BASICS_PATH  = f"{BASE_DIR}/title.basics.tsv"
RATINGS_PATH = f"{BASE_DIR}/title.ratings.tsv"
AKAS_PATH    = f"{BASE_DIR}/title.akas.tsv"
CREW_PATH    = f"{BASE_DIR}/title.crew.tsv"
NAMES_PATH = f"{BASE_DIR}/name.basics.tsv"
PRINCIPALS_PATH = f"{BASE_DIR}/title.principals.tsv"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import gc

## **Part 1. Load, Preprocess, and Clean Datasets**

### 1.1 `basics` dataset

In [4]:
basics_df = pd.read_csv(BASICS_PATH, sep='\t', na_values='\\N', low_memory=True)

/tmp/ipython-input-3018627348.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  basics_df = pd.read_csv(BASICS_PATH, sep='\t', na_values='\\N', low_memory=True)


In [5]:
basics_df.head(5)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894.0,NaN,1.0,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892.0,NaN,5.0,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892.0,NaN,5.0,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892.0,NaN,12.0,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893.0,NaN,1.0,Short


In [30]:
basics_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 652583 entries, 8 to 12029214
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   movieId         652583 non-null  object 
 1   movieType       652583 non-null  object 
 2   title           652583 non-null  object 
 3   isAdult         652583 non-null  int64  
 4   startYear       551472 non-null  float64
 5   runtimeMinutes  434705 non-null  float64
 6   genres          652583 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 39.8+ MB


Check null values for each features

In [29]:
basics_df.isnull().sum()

,0
movieId,0
movieType,0
title,0
isAdult,0
startYear,101111
runtimeMinutes,217878
genres,0


For our implementation, movie titles and genres are necessary. So we dropped the rows where `primaryTitle`, `originalTitle`, `genres` are null.

In [8]:
basics_df = basics_df.dropna(subset=['primaryTitle', 'originalTitle', 'genres'])

Drop unrelated columns and Rename some columns

In [9]:
basics_df = basics_df.drop(columns=['originalTitle', 'endYear'])
basics_df = basics_df.rename(columns={'primaryTitle': 'title', 'tconst': 'movieId'})

Keep only rows where `titleType == "movie"` and rename `titleType` as `movieType`

In [10]:
basics_df = basics_df[basics_df['titleType'] == 'movie']
basics_df = basics_df.rename(columns={'titleType': 'movieType'})

Make sure `startYear` and `runtimeMinutes` are numeric with NaN for missing values

In [11]:
basics_df["startYear"] = pd.to_numeric(basics_df["startYear"], errors="coerce")
basics_df["runtimeMinutes"] = pd.to_numeric(basics_df["runtimeMinutes"], errors="coerce")

Handle and explode `genres` Column

In [48]:
# Build genre_exploded once, cleanly, in Part 1

genre_exploded = basics_df[["movieId", "genres"]].copy()

# Split comma-separated genres into a list
genre_exploded["genres"] = genre_exploded["genres"].str.split(",")

# Explode into one row per (movieId, genre)
genre_exploded = genre_exploded.explode("genres", ignore_index=True)

# Drop missing genres (NaN after na_values="\\N") and empty strings
genre_exploded = genre_exploded.dropna(subset=["genres"])
genre_exploded["genres"] = genre_exploded["genres"].str.strip()
genre_exploded = genre_exploded[genre_exploded["genres"] != ""]

# Rename to 'genre'
genre_exploded = genre_exploded.rename(columns={"genres": "genre"}).reset_index(drop=True)

Make a copy of all movie ids

In [13]:
movie_ids = set(basics_df["movieId"].unique())

### 1.2 `ratings` dataset

In [14]:
ratings_df = pd.read_csv(RATINGS_PATH, sep='\t', na_values = '\\N', low_memory=True)

In [15]:
ratings_df.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2183
1,tt0000002,5.5,302
2,tt0000003,6.4,2263
3,tt0000004,5.2,194
4,tt0000005,6.2,3001


In [16]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1632106 entries, 0 to 1632105
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   tconst         1632106 non-null  object 
 1   averageRating  1632106 non-null  float64
 2   numVotes       1632106 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 37.4+ MB


In [17]:
ratings_df.isnull().sum()

,0
tconst,0
averageRating,0
numVotes,0


Rename columns to match our implementation

In [18]:
ratings_df = ratings_df.rename(columns={'tconst': 'movieId', 'averageRating': 'rating'})

Keep only ratings for movies

In [19]:
ratings_df = ratings_df[ratings_df['movieId'].isin(movie_ids)].copy()

### 1.3 `principal` dataset

In [20]:
# Build roles_df from title.principals.tsv
keep_categories = {"director", "writer", "actor", "actress"}

roles_chunks = []

for chunk in pd.read_csv(
    PRINCIPALS_PATH,
    sep="\t",
    usecols=["tconst", "nconst", "category"],  # only the columns we need
    na_values="\\N",
    low_memory=True,
    chunksize=5_000_000   # read 5M rows at a time to avoid RAM issues
):
    # Keep only rows for movies that are in our cleaned basics_df
    chunk = chunk[chunk["tconst"].isin(movie_ids)]

    # Keep only the role categories we care about
    chunk = chunk[chunk["category"].isin(keep_categories)]

    # Append this filtered chunk
    roles_chunks.append(chunk)

    # simple progress print
    print("Roles rows kept so far:", sum(len(c) for c in roles_chunks))

    # Concatenate all filtered chunks into a single DataFrame
roles_df = pd.concat(roles_chunks, ignore_index=True)

# Free memory used by the chunks list
del roles_chunks
gc.collect()

print("roles_df shape after concatenation:", roles_df.shape)
roles_df.head()

Roles rows kept so far: 1916520
Roles rows kept so far: 2036950
Roles rows kept so far: 2168949
Roles rows kept so far: 2314221
Roles rows kept so far: 2462789
Roles rows kept so far: 2653268
Roles rows kept so far: 2817418
Roles rows kept so far: 3008903
Roles rows kept so far: 3204501
Roles rows kept so far: 3421740
Roles rows kept so far: 3608587
Roles rows kept so far: 3776430
Roles rows kept so far: 3993275
Roles rows kept so far: 4198725
Roles rows kept so far: 4411357
Roles rows kept so far: 4664953
Roles rows kept so far: 4863450
Roles rows kept so far: 5019054
Roles rows kept so far: 5145704
Roles rows kept so far: 5165453
roles_df shape after concatenation: (5165453, 3)


,tconst,nconst,category
0,tt0000009,nm0063086,actress
1,tt0000009,nm0183823,actor
2,tt0000009,nm1309758,actor
3,tt0000009,nm0085156,director
4,tt0000009,nm0085156,writer


In [21]:
# Rename columns to intuitive names and keep only what we need
roles_df = roles_df.rename(columns={
    "tconst":  "movieId",      # match basics_df
    "nconst":  "person_id",    # clearer than nconst
    "category": "role"    # director / writer / actor / actress
})[["movieId", "person_id", "role"]].copy()

# drop exact duplicate rows
roles_df = roles_df.drop_duplicates()

print("roles_df shape after renaming & dropping duplicates:", roles_df.shape)
roles_df.head()

roles_df shape after renaming & dropping duplicates: (5099666, 3)


,movieId,person_id,role
0,tt0000009,nm0063086,actress
1,tt0000009,nm0183823,actor
2,tt0000009,nm1309758,actor
3,tt0000009,nm0085156,director
4,tt0000009,nm0085156,writer


In [22]:
roles_df.isna().sum()

,0
movieId,0
person_id,0
role,0


In [23]:
# Get all unique person IDs from roles_df
people_ids_set = set(roles_df["person_id"].unique())
print("Number of unique people IDs in roles_df:", len(people_ids_set))

Number of unique people IDs in roles_df: 1600297


### 1.4 `name.basics` dataset

In [24]:
# Load in by only keep the columns we want
people_raw_df = pd.read_csv(
    NAMES_PATH,
    sep="\t",
    usecols=["nconst", "primaryName", "birthYear", "deathYear", "knownForTitles"],
    na_values="\\N",
    low_memory=True
)

print("people_raw_df shape (all people):", people_raw_df.shape)
people_raw_df.head()

# Filter to only people that appear in roles_df
people_df = people_raw_df[people_raw_df["nconst"].isin(people_ids_set)].copy()
print("people_df shape after filtering to roles_df people:", people_df.shape)

people_raw_df shape (all people): (14841544, 5)
people_df shape after filtering to roles_df people: (1599334, 5)


In [25]:
people_df.isna().sum()

,0
nconst,0
primaryName,1
birthYear,1316603
deathYear,1487932
knownForTitles,6830


Drop the row without primaryName

In [26]:
people_df = people_df.dropna(subset=['primaryName'])

Convert feature type, rename columns

In [27]:
# Convert years to numeric
people_df["birthYear"] = pd.to_numeric(people_df["birthYear"], errors="coerce")
people_df["deathYear"] = pd.to_numeric(people_df["deathYear"], errors="coerce")

# Rename columns:
# - nconst        -> person_id  (to match roles_df)
# - primaryName   -> name
people_df = people_df.rename(columns={
    "nconst": "person_id",
    "primaryName": "name"
})
people_df.head()

,person_id,name,birthYear,deathYear,knownForTitles
0,nm0000001,Fred Astaire,1899.0,1987.0,"tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924.0,2014.0,"tt0037382,tt0075213,tt0038355,tt0117057"
2,nm0000003,Brigitte Bardot,1934.0,NaN,"tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949.0,1982.0,"tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,1918.0,2007.0,"tt0050986,tt0069467,tt0083922,tt0050976"


Explode knownForTitles column

In [31]:
known_for_df = people_df[["person_id", "knownForTitles"]].copy()

# Drop rows with no knownForTitles (optional, but cleaner)
known_for_df = known_for_df.dropna(subset=["knownForTitles"])

# Ensure it's string, then split
known_for_df["knownForTitles"] = known_for_df["knownForTitles"].astype(str)
known_for_df["movieId_list"] = known_for_df["knownForTitles"].str.split(",")

person_known_for_df = (
    known_for_df
    .explode("movieId_list", ignore_index=True)
    [["person_id", "movieId_list"]]
    .rename(columns={"movieId_list": "movieId"})
    .drop_duplicates()
)

print("person_known_for_df shape:", person_known_for_df.shape)
person_known_for_df.head()

person_known_for_df shape: (4325660, 2)


,person_id,movieId
0,nm0000001,tt0072308
1,nm0000001,tt0050419
2,nm0000001,tt0027125
3,nm0000001,tt0025164
4,nm0000002,tt0037382


## **Part 2. Entity Resolution**

### 2.1 Define final set of movies

For our project, we want the website to:

- Support **searching movies by name**, even if they don’t have any rating yet.
- Use **ratings** only when needed (for ranking / recommendations).

So instead of restricting ourselves to the intersection of `title.basics` and `title.ratings`, we define our **final movie universe** as:

> **All movies that pass our cleaning in `basics_df`.**

Then we treat `ratings_df` as **optional** metadata:
- If a movie has a rating, we’ll store it and use it in recommendation / “top movies” queries.
- If a movie does **not** have a rating, it still exists in our database and can be searched by title, year, genre, etc., but it won’t show up in rating-based rankings.


In [40]:
basics_movie_ids  = set(basics_df["movieId"].unique())
ratings_movie_ids = set(ratings_df["movieId"].unique())

final_movie_ids = basics_movie_ids

print("Movies in basics_df:", len(basics_movie_ids))
print("Movies in ratings_df:", len(ratings_movie_ids))
print("Movies in FINAL universe:", len(final_movie_ids))

Movies in basics_df: 652583
Movies in ratings_df: 326554
Movies in FINAL universe: 652583


### 2.2 Filter Other Datasets to the Final Movie Universe

In Section 2.1, we defined our **final movie universe** as:

> All movies that passed our cleaning in `basics_df`.

This means:

- Every movie in our system **must** appear in `basics_df`.
- Other datasets (`ratings_df`, `roles_df`, `person_known_for_df`) should only reference these movies.

So in this step, we:

- **Do not** drop any rows from `basics_df`.  
- **Do** filter:
  - `ratings_df` → keep only rows where `movieId ∈ final_movie_ids`  
  - `roles_df` → keep only rows where `movieId ∈ final_movie_ids`  
  - `person_known_for_df` → keep only rows where `movieId ∈ final_movie_ids`
  - `genre_exploded` – one row per (movieId, genre) that we will later use to build `Genres` and `MovieGenres`

This enforces consistent entity resolution on the **movie** side:  
- No rating, role, known-for link, or genre row will point to a movie that doesn’t exist in our final `basics_df` universe.

In [47]:
# Filter ratings_df to movies that exist in basics_df
before = len(ratings_df)
ratings_df = ratings_df[ratings_df["movieId"].isin(final_movie_ids)].copy()
after = len(ratings_df)
print(f"ratings_df: {before} -> {after} rows after movie filter")

# Filter roles_df to movies that exist in basics_df
before = len(roles_df)
roles_df = roles_df[roles_df["movieId"].isin(final_movie_ids)].copy()
after = len(roles_df)
print(f"roles_df: {before} -> {after} rows after movie filter")

# Filter person_known_for_df to movies that exist in basics_df (if it's defined)
before = len(person_known_for_df)
person_known_for_df = person_known_for_df[
    person_known_for_df["movieId"].isin(final_movie_ids)
].copy()
after = len(person_known_for_df)
print(f"person_known_for_df: {before} -> {after} rows after movie filter")

# Filter genre_exploded to movies that exist in basics_df
before = len(genre_exploded)
genre_exploded = genre_exploded[genre_exploded["movieId"].isin(final_movie_ids)].copy()
after = len(genre_exploded)
print(f"genre_exploded: {before} -> {after} rows after movie filter")

ratings_df: 326554 -> 326554 rows after movie filter
roles_df: 5098633 -> 5098633 rows after movie filter
person_known_for_df: 2907191 -> 2907191 rows after movie filter
genre_exploded: 1000097 -> 1000097 rows after movie filter


### 2.3 Define Final Person Universe and Filter People

Now that the movie side is consistent (only movieIds that exist in `basics_df`), we do the same for **people**.

Goals:

- **Every** `person_id` that appears in:
  - `roles_df` (directors / writers / actors / actresses), and  
  - `person_known_for_df`
  
  must have a corresponding row in `people_df`.
- We treat a person's **name** as a core attribute:
  - If we are missing the name, we drop that person from `people_df` and remove any roles/known-for rows that reference them.

This ensures:
- No foreign key (`person_id` in Roles / PersonKnownFor) points to a non-existent person.
- Every person shown in the UI has at least a valid `name`.


In [43]:
# People that actually appear in roles OR in person_known_for_df

role_person_ids     = set(roles_df["person_id"].unique())
knownfor_person_ids = set(person_known_for_df["person_id"].unique())
used_person_ids = role_person_ids | knownfor_person_ids

print("People in roles_df:", len(role_person_ids))
print("People in person_known_for_df:", len(knownfor_person_ids))
print("People in FINAL person universe (used somewhere):", len(used_person_ids))

# Filter people_df to only used people
before = len(people_df)
people_df = people_df[people_df["person_id"].isin(used_person_ids)].copy()
after = len(people_df)
print(f"people_df: {before} -> {after} rows after restricting to used people")

# Require name as a core attribute (drop nameless people)
before = len(people_df)
people_df = people_df.dropna(subset=["name"]).copy()
after = len(people_df)
print(f"people_df: {before} -> {after} rows after dropping nameless people")

# Enforce that roles / known_for only reference existing people
final_person_ids = set(people_df["person_id"].unique())

# Filter roles_df on person side
before = len(roles_df)
roles_df = roles_df[roles_df["person_id"].isin(final_person_ids)].copy()
after = len(roles_df)
print(f"roles_df: {before} -> {after} rows after enforcing people_df")

# Filter person_known_for_df on person side
before = len(person_known_for_df)
person_known_for_df = person_known_for_df[
    person_known_for_df["person_id"].isin(final_person_ids)
].copy()
after = len(person_known_for_df)
print(f"person_known_for_df: {before} -> {after} rows after enforcing people_df")


People in roles_df: 1600297
People in person_known_for_df: 1544294
People in FINAL person universe (used somewhere): 1600297
people_df: 1599333 -> 1599333 rows after restricting to used people
people_df: 1599333 -> 1599333 rows after dropping nameless people
roles_df: 5099666 -> 5098633 rows after enforcing people_df
person_known_for_df: 2907191 -> 2907191 rows after enforcing people_df


## **Part 3. Build Final Tables**

### 3.1 Movies Table

Goal: build a `movies_df` DataFrame that maps directly to the final `Movies` SQL table.

We want one row per movie with:

- `movie_id` (primary key, from `movieId`)
- `title` (movie title)
- `release_year` (from `startYear`)
- `runtime_minutes` (from `runtimeMinutes`)
- `is_adult` (0/1)
- `rating` (from `ratings_df`, may be NULL)
- `num_votes` (from `ratings_df`, may be NULL)

**Key design choice (Option B):**

- We keep **all movies from `basics_df`**.
- We **LEFT JOIN** ratings, so `rating` / `num_votes` can be missing.
- This lets the web app:
  - show all movies in title search, and
  - still use rating-based queries only on rows where `rating IS NOT NULL`.


In [44]:
# LEFT join: keep all movies from basics_df, add ratings when available
movies_df = basics_df.merge(
    ratings_df[["movieId", "rating", "numVotes"]],
    on="movieId",
    how="left"
)

# Rename columns to final, DB-friendly names
movies_df = movies_df.rename(columns={
    "movieId": "movie_id",
    "title": "title",
    "startYear": "release_year",
    "runtimeMinutes": "runtime_minutes",
    "isAdult": "is_adult",
    "numVotes": "num_votes"
})

# Keep only the columns we want in the Movies table
movies_df = movies_df[[
    "movie_id",
    "title",
    "release_year",
    "runtime_minutes",
    "is_adult",
    "rating",
    "num_votes"
]]

print("movies_df shape:", movies_df.shape)
movies_df.head()


movies_df shape: (652583, 7)


,movie_id,title,release_year,runtime_minutes,is_adult,rating,num_votes
0,tt0000009,Miss Jerry,1894.0,45.0,0,5.3,230.0
1,tt0000147,The Corbett-Fitzsimmons Fight,1897.0,100.0,0,5.3,575.0
2,tt0000335,Soldiers of the Cross,1900.0,40.0,0,5.5,62.0
3,tt0000574,The Story of the Kelly Gang,1906.0,70.0,0,6.0,1034.0
4,tt0000591,The Prodigal Son,1907.0,90.0,0,5.3,34.0


In [45]:
# Unique Index check
duplicate_movie_ids = movies_df[movies_df.duplicated(subset=["movie_id"], keep=False)]
print(duplicate_movie_ids.empty)


True


### 3.2 Genres and MovieGenres Tables

Now we normalize genres into two tables:

- **Genres**
  - `genre_id` (primary key)
  - `genre_name`

- **MovieGenres**
  - `movie_id` (FK → Movies.movie_id)
  - `genre_id` (FK → Genres.genre_id)
  - Composite primary key: (`movie_id`, `genre_id`)

Steps:
1. Take the unique genre names from `genre_exploded["genre"]` and assign them integer IDs.
2. Map each `(movieId, genre)` row to `(movie_id, genre_id)` to form the `MovieGenres` bridge.
3. Check that:
   - `genre_id` is unique in `Genres`.
   - `(movie_id, genre_id)` pairs are unique in `MovieGenres`.


In [50]:
# Pair categorical unique genre names with genre_id

unique_genres = sorted(genre_exploded["genre"].unique().tolist())

genres_df = pd.DataFrame({
    "genre_id": range(1, len(unique_genres) + 1),
    "genre_name": unique_genres
})

print("genres_df shape:", genres_df.shape)
display(genres_df.head())


genres_df shape: (28, 2)


,genre_id,genre_name
0,1,Action
1,2,Adult
2,3,Adventure
3,4,Animation
4,5,Biography


In [51]:
# Build MovieGenres bridge table by mapping genre_name -> genre_id

# Create mapping dictionary: genre_name -> genre_id
genre_map = dict(zip(genres_df["genre_name"], genres_df["genre_id"]))

# Start from (movieId, genre) pairs
movie_genres_df = genre_exploded[["movieId", "genre"]].copy()

# Map genre to genre_id
movie_genres_df["genre_id"] = movie_genres_df["genre"].map(genre_map)

# Rename movieId -> movie_id and keep only final columns
movie_genres_df = movie_genres_df.rename(columns={
    "movieId": "movie_id"
})[["movie_id", "genre_id"]].drop_duplicates()

print("movie_genres_df shape:", movie_genres_df.shape)
display(movie_genres_df.head())


movie_genres_df shape: (1000097, 2)


,movie_id,genre_id
0,tt0000009,21
1,tt0000147,8
2,tt0000147,19
3,tt0000147,24
4,tt0000335,5


In [52]:
# Uniqueness checks for PKs / indexes

# genre_id must be unique in Genres
dup_genre_ids = genres_df[genres_df.duplicated(subset=["genre_id"], keep=False)]
print(dup_genre_ids.empty)

# (movie_id, genre_id) pairs must be unique in MovieGenres
dup_movie_genres = movie_genres_df[
    movie_genres_df.duplicated(subset=["movie_id", "genre_id"], keep=False)
]
print(dup_movie_genres.empty)


True
True


### 3.3 People Table

The **People** table stores information about individuals (actors, actresses, directors, writers) who are associated with at least one movie in our final dataset.

After entity resolution in Section 2, `people_df` contains only:
- people who appear in `roles_df` and/or `person_known_for_df`, and  
- rows where `name` is not null.

We then construct the final **People** table with the following schema:

- `person_id` – primary key (IMDb `nconst`), unique identifier for each person  
- `name` – the person’s primary credited name  
- `birth_year` – year of birth (nullable)  
- `death_year` – year of death (nullable)

The original IMDb field `knownForTitles` is **not** stored in this table. It is a multi-valued attribute, so we normalize it into a separate `PersonKnownFor(person_id, movie_id)` table.

In [58]:
person_known_for_df_final = person_known_for_df.copy()

# Rename movieId -> movie_id and select final columns
person_known_for_df_final = person_known_for_df_final.rename(columns={
    "movieId": "movie_id"
})[["person_id", "movie_id"]].drop_duplicates()

print("person_known_for_df_final shape:", person_known_for_df_final.shape)
person_known_for_df_final.head()

person_known_for_df_final shape: (2907191, 2)


,person_id,movie_id
0,nm0000001,tt0072308
1,nm0000001,tt0050419
2,nm0000001,tt0027125
3,nm0000001,tt0025164
4,nm0000002,tt0037382


In [59]:
# Composite key check
dup_pkf = person_known_for_df_final[
    person_known_for_df_final.duplicated(subset=["person_id", "movie_id"], keep=False)
]
print(dup_pkf.empty)


True


In [60]:
import os

# ============================================================
# 1. Set export directory in your Shared Drive
# ============================================================

AWS_EXPORT_DIR = "/content/drive/Shareddrives/CIS5500/Cleaned_Datasets"
os.makedirs(AWS_EXPORT_DIR, exist_ok=True)
print("Export directory:", AWS_EXPORT_DIR)

# ============================================================
# 2. Rebuild final tables from current DataFrames
#    (assumes you've already run all preprocessing)
# ============================================================

# --- 2.1 People (People table) ---
people_df_final = people_df.rename(columns={
    "birthYear": "birth_year",
    "deathYear": "death_year"
})[["person_id", "name", "birth_year", "death_year"]].copy()

print("people_df_final:", people_df_final.shape)

# --- 2.2 Roles (Roles table) ---
roles_df_final = roles_df.rename(columns={
    "movieId": "movie_id"
})[["movie_id", "person_id", "role"]].drop_duplicates()

print("roles_df_final:", roles_df_final.shape)

# --- 2.3 PersonKnownFor (PersonKnownFor table) ---
person_known_for_df_final = person_known_for_df.copy()

valid_movie_ids = set(movies_df["movie_id"])
person_known_for_df_final = person_known_for_df_final[
    person_known_for_df_final["movieId"].isin(valid_movie_ids)
].copy()

person_known_for_df_final = (
    person_known_for_df_final
    .rename(columns={"movieId": "movie_id"})
    [["person_id", "movie_id"]]
    .drop_duplicates()
)

print("person_known_for_df_final:", person_known_for_df_final.shape)

# ============================================================
# 3. Export all tables to CSV into Cleaned_Datasets
# ============================================================

movies_path           = os.path.join(AWS_EXPORT_DIR, "movies.csv")
genres_path           = os.path.join(AWS_EXPORT_DIR, "genres.csv")
movie_genres_path     = os.path.join(AWS_EXPORT_DIR, "movie_genres.csv")
people_path           = os.path.join(AWS_EXPORT_DIR, "people.csv")
roles_path            = os.path.join(AWS_EXPORT_DIR, "roles.csv")
person_known_for_path = os.path.join(AWS_EXPORT_DIR, "person_known_for.csv")

movies_df.to_csv(movies_path, index=False)
genres_df.to_csv(genres_path, index=False)
movie_genres_df.to_csv(movie_genres_path, index=False)
people_df_final.to_csv(people_path, index=False)
roles_df_final.to_csv(roles_path, index=False)
person_known_for_df_final.to_csv(person_known_for_path, index=False)

print("Saved CSV files to:")
print("  ", movies_path)
print("  ", genres_path)
print("  ", movie_genres_path)
print("  ", people_path)
print("  ", roles_path)
print("  ", person_known_for_path)


Export directory: /content/drive/Shareddrives/CIS5500/Cleaned_Datasets
people_df_final: (1599333, 4)
roles_df_final: (5098633, 3)
person_known_for_df_final: (2907191, 2)
Saved CSV files to:
   /content/drive/Shareddrives/CIS5500/Cleaned_Datasets/movies.csv
   /content/drive/Shareddrives/CIS5500/Cleaned_Datasets/genres.csv
   /content/drive/Shareddrives/CIS5500/Cleaned_Datasets/movie_genres.csv
   /content/drive/Shareddrives/CIS5500/Cleaned_Datasets/people.csv
   /content/drive/Shareddrives/CIS5500/Cleaned_Datasets/roles.csv
   /content/drive/Shareddrives/CIS5500/Cleaned_Datasets/person_known_for.csv
